In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [10]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "full_under_200" in file:
            path = os.path.join(root, file)
            print("FOUND:", path)
            print("SIZE:", os.path.getsize(path) / (1024**2), "MB")



FOUND: /kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt
SIZE: 166.57985973358154 MB


In [11]:
DATA_PATH = "/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"

with open(DATA_PATH, "r") as f:
    text = f.read()

print("[ID] =", text.count("[ID]"))
print("[PRIMARY] =", text.count("[PRIMARY]"))
print("[EVOLUTIONARY] =", text.count("[EVOLUTIONARY]"))
print("[TERTIARY] =", text.count("[TERTIARY]"))
print("[DIST] =", text.count("[DIST]"))


[ID] = 636
[PRIMARY] = 636
[EVOLUTIONARY] = 636
[TERTIARY] = 636
[DIST] = 636


In [12]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [13]:
!git clone https://github.com/hypnopump/MiniFold.git

Cloning into 'MiniFold'...
remote: Enumerating objects: 400, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 400 (delta 69), reused 126 (delta 54), pack-reused 255 (from 1)
Receiving objects: 100% (400/400), 35.53 MiB | 36.68 MiB/s, done.
Resolving deltas: 100% (187/187), done.


In [15]:
import os

print("Searching for MiniFold...")

for root, dirs, files in os.walk("/kaggle/working"):
    if "func_utils.py" in files:
        print("FOUND func_utils.py:")
        print(os.path.join(root, "func_utils.py"))


Searching for MiniFold...
FOUND func_utils.py:
/kaggle/working/MiniFold/models/distance_pipeline/func_utils.py


In [16]:
import os

BASE = "/kaggle/working/MiniFold/models/distance_pipeline"

print("Files:")
for f in os.listdir(BASE):
    print(f)


Files:
pretrain_model_pssm_l_x_l.ipynb
models
evaluation_pipeline.py
distance_generator_data.py
elu_resnet_2d_distances.py
record.txt
images
func_utils.py
pipeline_caller.py
training_pipeline.py
Tutorials


In [17]:
BASE = "/kaggle/working/MiniFold/models/distance_pipeline"

with open(os.path.join(BASE, "func_utils.py"), "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines[:70], start=1):
    print(f"{i:3}: {line.rstrip()}")


  1: """ File to store functions (DRY) and params that are repeated
  2:     Store config only in one site
  3: """
  4: import numpy as np
  5: import keras
  6: # Logs-related imports
  7: import sys
  8: import logging
  9: import os
 10: 
 11: # Logging info: - For reproducibility //
 12: # np.random.seed(5)
 13: 
 14: # PATHS
 15: log_name = str(sys.argv[1])
 16: LOG_PATH = str(os.getcwd())+"/"+log_name+".txt"
 17: RECORD_PATH = "record.txt"
 18: BASE_MODEL_PATH = "models/tester_28_lxl.h5"
 19: STAGE_MODEL_PATH = "models/tester_28_lxl_stage.h5"
 20: GOLDEN_MODEL_PATH = "models/tester_28_lxl_golden.h5"
 21: IMAGES_PATH = "images/golden_img_v"+str(sys.argv[2])+"_"
 22: TRAINING_PATHS = ["../../data/distanced/90_full_under_200.txt"]
 23:                 # "../data/distanced/70_full_under_200.txt"]
 24: EVAL_SOURCE_PATH = "../../data/distanced/full_under_200.txt"
 25: 
 26: # PARAMS - DEFINE MODEL PARAMS
 27: CROP_SIZE = 200                                      # Crops to feed the mmo

In [18]:
import shutil

BASE = "/kaggle/working/MiniFold/models/distance_pipeline"

shutil.copy(
    f"{BASE}/func_utils.py",
    f"{BASE}/func_utils.py.original"
)

print("Backup created")


Backup created


In [19]:
from pathlib import Path

BASE = Path("/kaggle/working/MiniFold/models/distance_pipeline")

path = BASE / "func_utils.py"

text = path.read_text()

# 1. Notebook 不使用 sys.argv
text = text.replace(
    'log_name = str(sys.argv[1])',
    'log_name = "kaggle_training"'
)

text = text.replace(
    'IMAGES_PATH = "images/golden_img_v"+str(sys.argv[2])+"_"',
    'IMAGES_PATH = "images/golden_img_"'
)

# 2. 使用 Kaggle 的 636-protein dataset
text = text.replace(
    'TRAINING_PATHS = ["../../data/distanced/90_full_under_200.txt"]',
    'TRAINING_PATHS = ["/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"]'
)

text = text.replace(
    'EVAL_SOURCE_PATH = "../../data/distanced/full_under_200.txt"',
    'EVAL_SOURCE_PATH = "/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"'
)

# 3. 636 proteins
text = text.replace(
    'MAX_PROTS = 1500 # 3500',
    'MAX_PROTS = 636'
)

# 4. Notebook 直接指定 class weights
old = '''print(str(sys.argv[3]).split(","))
WEIGHTS = [float(w) for w in str(sys.argv[3]).split(",")] # Class weights for the model'''

new = '''WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0] # Class weights'''

text = text.replace(old, new)

# 5. Keras 3 optimizer syntax
text = text.replace(
    'keras.optimizers.Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=None, decay=0.0, amsgrad=True)',
    'keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=None, amsgrad=True)'
)

path.write_text(text)

print("func_utils.py updated")


func_utils.py updated


In [20]:
with open(BASE / "func_utils.py", "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines[14:56], start=15):
    print(f"{i:3}: {line.rstrip()}")


 15: log_name = "kaggle_training"
 16: LOG_PATH = str(os.getcwd())+"/"+log_name+".txt"
 17: RECORD_PATH = "record.txt"
 18: BASE_MODEL_PATH = "models/tester_28_lxl.h5"
 19: STAGE_MODEL_PATH = "models/tester_28_lxl_stage.h5"
 20: GOLDEN_MODEL_PATH = "models/tester_28_lxl_golden.h5"
 21: IMAGES_PATH = "images/golden_img_"
 22: TRAINING_PATHS = ["/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"]
 23:                 # "../data/distanced/70_full_under_200.txt"]
 24: EVAL_SOURCE_PATH = "/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"
 25: 
 26: # PARAMS - DEFINE MODEL PARAMS
 27: CROP_SIZE = 200                                      # Crops to feed the mmodel
 28: PAD_SIZE = 200                                       # Maximum length of any protein
 29: CLASS_CUTS = [-0.5, 500, 750, 1000, 1400, 1700] # , 2000 # Cuts between classes
 30: N_CLASSES = len(CLASS_CUTS)+1                        # Number of classes
 31: BATCH_SIZE = 2               

In [21]:
import re

MODEL_FILE = BASE / "elu_resnet_2d_distances.py"

text = MODEL_FILE.read_text()

for line in text.splitlines():
    if "def softMaxAxis2" in line or "return softmax" in line:
        print(line)


def softMaxAxis2(x):
    return softmax(x,axis=2)


In [23]:
from pathlib import Path

BASE = Path("/kaggle/working/MiniFold/models/distance_pipeline")
FUNC_UTILS = BASE / "func_utils.py"

text = FUNC_UTILS.read_text()

# ----------------------------------------
# Kaggle paths
# ----------------------------------------

text = text.replace(
    'log_name = str(sys.argv[1])',
    'log_name = "kaggle_training"'
)

text = text.replace(
    'IMAGES_PATH = "images/golden_img_v"+str(sys.argv[2])+"_"',
    'IMAGES_PATH = "images/golden_img_"'
)

text = text.replace(
    'TRAINING_PATHS = ["../../data/distanced/90_full_under_200.txt"]',
    'TRAINING_PATHS = ["/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"]'
)

text = text.replace(
    'EVAL_SOURCE_PATH = "../../data/distanced/full_under_200.txt"',
    'EVAL_SOURCE_PATH = "/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"'
)

# ----------------------------------------
# Train all 636 proteins
# ----------------------------------------

text = text.replace(
    'MAX_PROTS = 1500 # 3500',
    'MAX_PROTS = 636'
)

text = text.replace(
    'BATCH_RATIO = 0.1 # Proportion of batches/prots (don\'t train on whole dataset)',
    'BATCH_RATIO = 1.0 # Train on the full dataset'
)

# ----------------------------------------
# Remove command-line WEIGHTS
# ----------------------------------------

old_weights = '''print(str(sys.argv[3]).split(","))
WEIGHTS = [float(w) for w in str(sys.argv[3]).split(",")] # Class weights for the model'''

new_weights = '''WEIGHTS = [
    1.0, 1.0, 1.0, 1.0,
    1.0, 1.0, 1.0
] # Class weights for the model'''

text = text.replace(old_weights, new_weights)

# ----------------------------------------
# Modern Keras Adam
# ----------------------------------------

text = text.replace(
    'keras.optimizers.Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=None, decay=0.0, amsgrad=True)',
    'keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=None, amsgrad=True)'
)

# ----------------------------------------
# Save trained model to Kaggle working
# ----------------------------------------

text = text.replace(
    'STAGE_MODEL_PATH = "models/tester_28_lxl_stage.h5"',
    'STAGE_MODEL_PATH = "/kaggle/working/tester_28_full636_trained.h5"'
)

FUNC_UTILS.write_text(text)

print("func_utils.py updated successfully")


func_utils.py updated successfully


In [24]:
print(FUNC_UTILS.read_text())


""" File to store functions (DRY) and params that are repeated
    Store config only in one site
"""
import numpy as np 
import keras
# Logs-related imports
import sys
import logging
import os

# Logging info: - For reproducibility // 
# np.random.seed(5)

# PATHS
log_name = "kaggle_training"
LOG_PATH = str(os.getcwd())+"/"+log_name+".txt"
RECORD_PATH = "record.txt"
BASE_MODEL_PATH = "models/tester_28_lxl.h5"
STAGE_MODEL_PATH = "/kaggle/working/tester_28_full636_trained.h5"
GOLDEN_MODEL_PATH = "models/tester_28_lxl_golden.h5"
IMAGES_PATH = "images/golden_img_"
TRAINING_PATHS = ["/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"] 
                # "../data/distanced/70_full_under_200.txt"]
EVAL_SOURCE_PATH = "/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"

# PARAMS - DEFINE MODEL PARAMS
CROP_SIZE = 200                                      # Crops to feed the mmodel   
PAD_SIZE = 200                                       # Maximum lengt

In [25]:
from pathlib import Path

MODEL_PY = Path(
    "/kaggle/working/MiniFold/models/distance_pipeline/elu_resnet_2d_distances.py"
)

text = MODEL_PY.read_text()

# 修正 softmax axis
text = text.replace(
    "return softmax(x,axis=2)",
    "return softmax(x,axis=-1)"
)

# 如果原本有空格版本，也一起處理
text = text.replace(
    "return softmax(x, axis=2)",
    "return softmax(x, axis=-1)"
)

MODEL_PY.write_text(text)

print("elu_resnet_2d_distances.py updated successfully")


elu_resnet_2d_distances.py updated successfully


In [26]:
text = MODEL_PY.read_text()

start = text.find("def softMaxAxis2")

print(text[start:start+150])


def softMaxAxis2(x):
    """ Compute softmax on axis 2. """
    return softmax(x,axis=-1)


def weighted_categorical_crossentropy(weights):
    """
  


In [27]:
import sys

for name in [
    "func_utils",
    "elu_resnet_2d_distances",
    "distance_generator_data"
]:
    sys.modules.pop(name, None)

print("Old modules cleared")


Old modules cleared


In [28]:
import sys

sys.path.insert(
    0,
    "/kaggle/working/MiniFold/models/distance_pipeline"
)

from func_utils import *

print("MAX_PROTS =", MAX_PROTS)
print("BATCH_SIZE =", BATCH_SIZE)
print("BATCH_RATIO =", BATCH_RATIO)
print("N_CLASSES =", N_CLASSES)
print("WEIGHTS =", WEIGHTS)
print("TRAINING_PATHS =", TRAINING_PATHS)


MAX_PROTS = 636
BATCH_SIZE = 2
BATCH_RATIO = 1.0
N_CLASSES = 7
WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
TRAINING_PATHS = ['/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt']


I0000 00:00:1790612912.477154      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790612912.480061      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [30]:
%cd /kaggle/working/MiniFold/models/distance_pipeline

path = "elu_resnet_2d_distances.py"

with open(path, "r") as f:
    text = f.read()

text = text.replace(
    "from keras.layers.convolutional import Conv2D, Conv2DTranspose",
    "from keras.layers import Conv2D, Conv2DTranspose"
)

text = text.replace(
    "from keras.layers.pooling import MaxPooling2D, AveragePooling2D",
    "from keras.layers import MaxPooling2D, AveragePooling2D"
)

text = text.replace(
    "return softmax(x,axis=2)",
    "return softmax(x, axis=-1)"
)

with open(path, "w") as f:
    f.write(text)

print("elu_resnet_2d_distances.py updated")


/kaggle/working/MiniFold/models/distance_pipeline
elu_resnet_2d_distances.py updated


In [31]:
!grep -n -A3 -B2 "def softMaxAxis2" elu_resnet_2d_distances.py


13-
14-
15:def softMaxAxis2(x):
16-    """ Compute softmax on axis 2. """
17-    return softmax(x,axis=-1)
18-


In [32]:
%cd /kaggle/working/MiniFold/models/distance_pipeline

path = "func_utils.py"

with open(path, "r") as f:
    text = f.read()

# sys.argv 設定改掉
text = text.replace(
    'log_name = str(sys.argv[1])',
    'log_name = "kaggle_training"'
)

text = text.replace(
    'IMAGES_PATH = "images/golden_img_v"+str(sys.argv[2])+"_"',
    'IMAGES_PATH = "images/golden_img_"'
)

text = text.replace(
    'TRAINING_PATHS = ["../../data/distanced/90_full_under_200.txt"]',
    'TRAINING_PATHS = ["/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"]'
)

text = text.replace(
    'EVAL_SOURCE_PATH = "../../data/distanced/full_under_200.txt"',
    'EVAL_SOURCE_PATH = "/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt"'
)

# dataset parameters
text = text.replace(
    'BATCH_SIZE = 2',
    'BATCH_SIZE = 2'
)

text = text.replace(
    'MAX_PROTS = 1500',
    'MAX_PROTS = 636'
)

text = text.replace(
    'BATCH_RATIO = 0.1',
    'BATCH_RATIO = 1.0'
)

# class weights
import re

text = re.sub(
    r'WEIGHTS\s*=\s*\[.*?\]',
    'WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]',
    text,
    count=1
)

# Keras 3 optimizer
text = re.sub(
    r'keras\.optimizers\.Adam\([^)]*\)',
    'keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=None, amsgrad=True)',
    text,
    count=1
)

# Remove old command-line print / WEIGHTS parsing
text = re.sub(
    r'print\(str\(sys\.argv\[3\]\)\.split\(","\)\)\s*\n',
    '',
    text
)

text = re.sub(
    r'WEIGHTS\s*=\s*\[float\(w\).*?\n',
    'WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]\n',
    text,
    count=1
)

with open(path, "w") as f:
    f.write(text)

print("func_utils.py updated")


/kaggle/working/MiniFold/models/distance_pipeline
func_utils.py updated


In [33]:
import sys

for name in [
    "func_utils",
    "elu_resnet_2d_distances",
    "distance_generator_data"
]:
    sys.modules.pop(name, None)

sys.path.insert(
    0,
    "/kaggle/working/MiniFold/models/distance_pipeline"
)

print("Python module cache cleared")


Python module cache cleared


In [34]:
from func_utils import (
    MAX_PROTS,
    BATCH_SIZE,
    BATCH_RATIO,
    N_CLASSES,
    WEIGHTS,
    TRAINING_PATHS
)

print("MAX_PROTS =", MAX_PROTS)
print("BATCH_SIZE =", BATCH_SIZE)
print("BATCH_RATIO =", BATCH_RATIO)
print("N_CLASSES =", N_CLASSES)
print("WEIGHTS =", WEIGHTS)
print("TRAINING_PATHS =", TRAINING_PATHS)


MAX_PROTS = 636
BATCH_SIZE = 2
BATCH_RATIO = 1.0
N_CLASSES = 7
WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
TRAINING_PATHS = ['/kaggle/input/datasets/timjas/minifold-full-under-200/full_under_200.txt']


In [35]:
from elu_resnet_2d_distances import softMaxAxis2

print("softMaxAxis2 imported successfully!")


softMaxAxis2 imported successfully!


In [36]:
from distance_generator_data import DataGenerator

print("DataGenerator imported successfully!")


DataGenerator imported successfully!


In [37]:
generator = DataGenerator(
    paths=TRAINING_PATHS,
    max_prots=MAX_PROTS,
    batch_size=BATCH_SIZE,
    crop_size=200,
    pad_size=200,
    n_classes=N_CLASSES,
    class_cuts=[-0.5, 500, 750, 1000, 1400, 1700],
    shuffle=False
)

print("Generator length =", len(generator))
print("Proteins =", len(generator.seqs))


Currently @  150  out of 636
Currently @  300  out of 636
Currently @  450  out of 636
Currently @  600  out of 636
Total length is 635 out of 636 possible.
Generator length = 318
Proteins = 636


In [39]:
%cd /kaggle/working/MiniFold/models/distance_pipeline

path = "distance_generator_data.py"

with open(path, "r") as f:
    text = f.read()

text = text.replace(".astype(np.int)", ".astype(int)")

with open(path, "w") as f:
    f.write(text)

print("distance_generator_data.py updated")


/kaggle/working/MiniFold/models/distance_pipeline
distance_generator_data.py updated


In [40]:
!grep -n "np.int" distance_generator_data.py

In [41]:
import sys

sys.modules.pop("distance_generator_data", None)

sys.path.insert(
    0,
    "/kaggle/working/MiniFold/models/distance_pipeline"
)

from distance_generator_data import DataGenerator

print("DataGenerator reloaded successfully!")


DataGenerator reloaded successfully!


In [42]:
generator = DataGenerator(
    paths=TRAINING_PATHS,
    max_prots=MAX_PROTS,
    batch_size=BATCH_SIZE,
    crop_size=200,
    pad_size=200,
    n_classes=N_CLASSES,
    class_cuts=[-0.5, 500, 750, 1000, 1400, 1700],
    shuffle=False
)

print("seqs =", len(generator.seqs))
print("dists =", len(generator.dists))
print("pssms =", len(generator.pssms))
print("generator length =", len(generator))


Currently @  150  out of 636
Currently @  300  out of 636
Currently @  450  out of 636
Currently @  600  out of 636
Total length is 635 out of 636 possible.
seqs = 636
dists = 636
pssms = 636
generator length = 318


In [43]:
x, y = generator[0]

print("x shape =", x.shape)
print("y shape =", y.shape)

print("x finite =", np.isfinite(x).all())
print("y finite =", np.isfinite(y).all())

print("y sum min =", y.sum(axis=-1).min())
print("y sum max =", y.sum(axis=-1).max())


x shape = (2, 200, 200, 42)
y shape = (2, 200, 200, 7)
x finite = True
y finite = True
y sum min = 1
y sum max = 1


In [44]:
import os
import sys
import numpy as np
import keras

sys.path.insert(0, "/kaggle/working/MiniFold/models/distance_pipeline")

from elu_resnet_2d_distances import (
    softMaxAxis2,
    weighted_categorical_crossentropy
)

MODEL_PATH = "/kaggle/working/MiniFold/models/distance_pipeline/models/tester_28_lxl_golden.h5"

model = keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "softMaxAxis2": softMaxAxis2
    },
    compile=False
)

print("Model loaded successfully!")
print("Input shape =", model.input_shape)
print("Output shape =", model.output_shape)


Model loaded successfully!
Input shape = (None, 200, 200, 42)
Output shape = (None, 200, 200, 7)


In [48]:
%cd /kaggle/working/MiniFold/models/distance_pipeline

path = "elu_resnet_2d_distances.py"

with open(path, "r") as f:
    text = f.read()

start = text.index("def weighted_categorical_crossentropy(weights):")
end = text.index("\ndef resnet_block(", start)

new_function = '''def weighted_categorical_crossentropy(weights):
    """Keras 3 compatible weighted categorical crossentropy."""

    weights = np.asarray(weights, dtype=np.float32)

    def loss(y_true, y_pred):
        # Normalize predictions across the class dimension.
        y_pred = y_pred / K.sum(
            y_pred, axis=-1, keepdims=True
        )

        # Prevent log(0).
        y_pred = K.clip(
            y_pred,
            K.epsilon(),
            1.0 - K.epsilon()
        )

        # Weighted categorical crossentropy.
        loss_value = (
            y_true
            * K.log(y_pred)
            * weights
        )

        return -K.sum(loss_value, axis=-1)

    return loss

'''

text = text[:start] + new_function + text[end + 1:]

with open(path, "w") as f:
    f.write(text)

print("weighted_categorical_crossentropy replaced successfully!")


/kaggle/working/MiniFold/models/distance_pipeline
weighted_categorical_crossentropy replaced successfully!


In [49]:
!python -m py_compile elu_resnet_2d_distances.py


In [50]:
!sed -n '15,60p' elu_resnet_2d_distances.py


def softMaxAxis2(x):
    """ Compute softmax on axis 2. """
    return softmax(x,axis=-1)


def weighted_categorical_crossentropy(weights):
    """Keras 3 compatible weighted categorical crossentropy."""

    weights = np.asarray(weights, dtype=np.float32)

    def loss(y_true, y_pred):
        # Normalize predictions across the class dimension.
        y_pred = y_pred / K.sum(
            y_pred, axis=-1, keepdims=True
        )

        # Prevent log(0).
        y_pred = K.clip(
            y_pred,
            K.epsilon(),
            1.0 - K.epsilon()
        )

        # Weighted categorical crossentropy.
        loss_value = (
            y_true
            * K.log(y_pred)
            * weights
        )

        return -K.sum(loss_value, axis=-1)

    return loss

def resnet_block(inputs,
                 num_filters=64,
                 kernel_size=3,
                 strides=1,
                 activation='elu',
                 batch_normalization=True,
                 conv_f

In [51]:
import sys

sys.modules.pop("elu_resnet_2d_distances", None)

from elu_resnet_2d_distances import (
    softMaxAxis2,
    weighted_categorical_crossentropy
)

print("Import OK")


Import OK


In [53]:
%cd /kaggle/working/MiniFold/models/distance_pipeline

from pathlib import Path

path = Path("elu_resnet_2d_distances.py")
text = path.read_text()

start = text.index("def weighted_categorical_crossentropy(weights):")
end = text.index("\ndef resnet_block", start)

new_func = '''def weighted_categorical_crossentropy(weights):
    """
    Keras 3 compatible weighted categorical cross entropy.
    """

    weights = np.asarray(weights, dtype=np.float32)

    def loss(y_true, y_pred):
        # Normalize probabilities along the class dimension.
        y_pred = y_pred / (
            keras.ops.sum(y_pred, axis=-1, keepdims=True)
            + keras.backend.epsilon()
        )

        # Avoid log(0).
        y_pred = keras.ops.clip(
            y_pred,
            keras.backend.epsilon(),
            1.0 - keras.backend.epsilon()
        )

        # Weighted categorical cross entropy.
        loss = (
            y_true
            * keras.ops.log(y_pred)
            * keras.ops.convert_to_tensor(weights)
        )

        return -keras.ops.sum(loss, axis=-1)

    return loss

'''

text = text[:start] + new_func + text[end:]

path.write_text(text)

print("weighted_categorical_crossentropy updated.")


/kaggle/working/MiniFold/models/distance_pipeline
weighted_categorical_crossentropy updated.


In [54]:
!sed -n '1,65p' elu_resnet_2d_distances.py


# Import libraries
import numpy as np
import keras
import keras.backend as K
from keras.models import Model
# Activation and Regularization
from keras.regularizers import l2
from keras.activations import softmax
# Keras layers
from keras.layers import Conv2D, Conv2DTranspose
from keras.layers import Dense, Dropout, Flatten, Input, BatchNormalization, Activation
from keras.layers import MaxPooling2D, AveragePooling2D


def softMaxAxis2(x):
    """ Compute softmax on axis 2. """
    return softmax(x,axis=-1)


def weighted_categorical_crossentropy(weights):
    """
    Keras 3 compatible weighted categorical cross entropy.
    """

    weights = np.asarray(weights, dtype=np.float32)

    def loss(y_true, y_pred):
        # Normalize probabilities along the class dimension.
        y_pred = y_pred / (
            keras.ops.sum(y_pred, axis=-1, keepdims=True)
            + keras.backend.epsilon()
        )

        # Avoid log(0).
        y_pred = keras.ops.clip(
            y_pred,
      

In [55]:
path = Path("distance_generator_data.py")
text = path.read_text()

text = text.replace(".astype(np.int)", ".astype(int)")

path.write_text(text)

print("np.int fixed.")


np.int fixed.


In [57]:
!grep -n "np.int" distance_generator_data.py


In [58]:
path = Path("elu_resnet_2d_distances.py")
text = path.read_text()

text = text.replace(
    "from keras.layers.convolutional import Conv2D, Conv2DTranspose",
    "from keras.layers import Conv2D, Conv2DTranspose"
)

text = text.replace(
    "from keras.layers.pooling import MaxPooling2D, AveragePooling2D",
    "from keras.layers import MaxPooling2D, AveragePooling2D"
)

path.write_text(text)

print("Keras 3 imports fixed.")


Keras 3 imports fixed.


In [59]:
import sys

for name in [
    "elu_resnet_2d_distances",
    "distance_generator_data",
    "func_utils"
]:
    sys.modules.pop(name, None)

sys.path.insert(0, "/kaggle/working/MiniFold/models/distance_pipeline")


In [60]:
from elu_resnet_2d_distances import (
    softMaxAxis2,
    weighted_categorical_crossentropy
)

print("imports OK")


imports OK


In [61]:
WEIGHTS = np.array([
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0
], dtype=np.float32)

loss_fn = weighted_categorical_crossentropy(WEIGHTS)

print("loss function ready")


loss function ready


In [63]:
loss_value = loss_fn(y, prediction)

print("loss shape =", loss_value.shape)
print("loss mean =", float(np.mean(loss_value)))
print("loss min =", float(np.min(loss_value)))
print("loss max =", float(np.max(loss_value)))


loss shape = (2, 200, 200)
loss mean = 1.6375987529754639
loss min = 7.629423635080457e-06
loss max = 9.164803504943848


In [62]:
prediction = model.predict(x, verbose=1)

print("prediction shape =", prediction.shape)

prediction_sum = np.sum(prediction, axis=-1)

print("prediction sum shape =", prediction_sum.shape)
print("min =", prediction_sum.min())
print("max =", prediction_sum.max())
print("mean =", prediction_sum.mean())


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
prediction shape = (2, 200, 200, 7)
prediction sum shape = (2, 200, 200)
min = 0.9999997
max = 1.0000002
mean = 1.0


In [65]:
print("Number of proteins =", len(generator.list_IDs))
print("Number of batches =", len(generator))
print("Batch size =", generator.batch_size)


Number of proteins = 636
Number of batches = 318
Batch size = 2


In [66]:
print("Input shape =", model.input_shape)
print("Output shape =", model.output_shape)


Input shape = (None, 200, 200, 42)
Output shape = (None, 200, 200, 7)


In [67]:
optimizer = keras.optimizers.Adam(
    learning_rate=0.001,
    beta_1=0.9,
    beta_2=0.999,
    amsgrad=True
)

loss_fn = weighted_categorical_crossentropy(
    np.array(WEIGHTS, dtype=np.float32)
)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=["accuracy"]
)

print("Model compiled successfully")


Model compiled successfully


In [68]:
history = model.fit(
    generator,
    epochs=1,
    verbose=1
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
2026-09-28 16:41:26.367438: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-28 16:41:26.644482: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


318/318 ━━━━━━━━━━━━━━━━━━━━ 364s 875ms/step - accuracy: 0.8721 - loss: 0.3983


In [69]:
OUTPUT_MODEL = "/kaggle/working/MiniFold_distance_trained.h5"

model.save(
    OUTPUT_MODEL,
    include_optimizer=True
)

print("Saved:", OUTPUT_MODEL)


Saved: /kaggle/working/MiniFold_distance_trained.h5


In [70]:
import os

print(
    "File exists =",
    os.path.exists(OUTPUT_MODEL)
)

print(
    "File size =",
    os.path.getsize(OUTPUT_MODEL) / (1024**2),
    "MB"
)


File exists = True
File size = 7.615531921386719 MB


In [71]:
reloaded_model = keras.models.load_model(
    OUTPUT_MODEL,
    custom_objects={
        "softMaxAxis2": softMaxAxis2,
        "loss": loss_fn
    },
    compile=False
)

print("Reloaded successfully!")
print("Input =", reloaded_model.input_shape)
print("Output =", reloaded_model.output_shape)


Reloaded successfully!
Input = (None, 200, 200, 42)
Output = (None, 200, 200, 7)


In [72]:
test_prediction = reloaded_model.predict(x, verbose=1)

test_sum = np.sum(test_prediction, axis=-1)

print("prediction shape =", test_prediction.shape)
print("sum min =", test_sum.min())
print("sum max =", test_sum.max())
print("sum mean =", test_sum.mean())


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
prediction shape = (2, 200, 200, 7)
sum min = 0.99999964
sum max = 1.0000004
sum mean = 1.0


In [73]:
!cd /kaggle/working && zip -r MiniFold_modified.zip MiniFold


  adding: MiniFold/ (stored 0%)
  adding: MiniFold/future.md (deflated 44%)
  adding: MiniFold/contributors.md (deflated 43%)
  adding: MiniFold/requirements.txt (deflated 47%)
  adding: MiniFold/.gitignore (deflated 31%)
  adding: MiniFold/models/ (stored 0%)
  adding: MiniFold/models/angles/ (stored 0%)
  adding: MiniFold/models/angles/resnet_1d_angles.h5 (deflated 13%)
  adding: MiniFold/models/angles/predicting_angles.ipynb (deflated 84%)
  adding: MiniFold/models/angles/resnet_1d_angles.py (deflated 69%)
  adding: MiniFold/models/new_distances/ (stored 0%)
  adding: MiniFold/models/new_distances/pretrain_model.ipynb (deflated 59%)
  adding: MiniFold/models/new_distances/resume_training.ipynb (deflated 57%)
  adding: MiniFold/models/new_distances/elu_resnet_2d_distances.py (deflated 70%)
  adding: MiniFold/models/new_distances/trained_models_h5/ (stored 0%)
  adding: MiniFold/models/new_distances/trained_models_h5/17_test.h5 (deflated 19%)
  adding: MiniFold/models/new_distances/tr